# 140. Word Break II

## Topic Alignment
- Word segmentation appears in NLP tokenization, domain name parsing, URL decomposition, and log parsing in ML systems.
- Memoization with backtracking is essential for generating all solutions in combinatorial problems like feature combination generation and hyperparameter grid search.
- The pattern of "find all solutions" vs "check if solution exists" distinguishes production debugging (need all) from optimization (need one).

## Metadata 摘要
- Source: https://leetcode.com/problems/word-break-ii/
- Tags: Dynamic Programming, Backtracking, Trie, Memoization, String
- Difficulty: Hard
- Priority: High

## Problem Statement 原题描述
Given a string `s` and a dictionary of strings `wordDict`, add spaces in `s` to construct a sentence where each word is a valid dictionary word. Return all such possible sentences in **any order**.

**Note** that the same word in the dictionary may be reused multiple times in the segmentation.

**Constraints**:
- `1 <= s.length <= 20`
- `1 <= wordDict.length <= 1000`
- `1 <= wordDict[i].length <= 10`
- `s` and `wordDict[i]` consist of only lowercase English letters.
- All the strings of `wordDict` are **unique**.

## Progressive Hints
- Hint 1: This is Word Break I but we need all solutions, not just True/False.
- Hint 2: Use backtracking to explore all possible segmentations.
- Hint 3: Memoize results for each starting position to avoid recomputation.
- Hint 4: `memo[i]` = list of all valid segmentations starting from index i.
- Hint 5: For each position, try all words that match the prefix.
- Hint 6: Recursively solve for remaining substring and combine results.
- Hint 7: Convert wordDict to set for O(1) lookup.
- Hint 8: Base case: empty string returns [""]; impossible returns [].

## Solution Overview
Use **backtracking with memoization (top-down DP)**.

**State Definition**:
- `memo[i]`: list of all valid sentence segmentations for `s[i:]`

**Recurrence**:
```python
def dfs(start):
    if start == len(s):
        return [""]
    
    if start in memo:
        return memo[start]
    
    results = []
    for end in range(start+1, len(s)+1):
        word = s[start:end]
        if word in wordDict:
            for sentence in dfs(end):
                if sentence:
                    results.append(word + " " + sentence)
                else:
                    results.append(word)
    
    memo[start] = results
    return results
```

**Optimization**: Check if segmentation is possible using Word Break I first to avoid unnecessary work.

## Detailed Explanation

### Difference from Word Break I

**Word Break I**: Return True/False
- Single boolean value
- Early termination when found one solution

**Word Break II**: Return all possible segmentations
- List of strings
- Must explore all branches
- Much higher time complexity

---

### Backtracking Strategy

**Core idea**: For each position, try all possible words that could start there.

```
s = "catsanddog"
wordDict = {"cat", "cats", "and", "sand", "dog"}

At position 0:
  Try "c" - not in dict
  Try "ca" - not in dict
  Try "cat" - in dict! → recurse on "sanddog"
  Try "cats" - in dict! → recurse on "anddog"
  ...
```

---

### Memoization Design

**Without memoization**:
```
s = "aaaaaaa"
wordDict = {"a", "aa", "aaa"}

Computing from position 2 multiple times:
- Path: "a" + "a" + ... (recurse from 2)
- Path: "aa" + ... (recurse from 2)
Exponential redundancy!
```

**With memoization**:
```python
memo[2] = ["a a a a a", "aa a a a", "a aa a a", ...]
# Compute once, reuse everywhere
```

---

### Combining Results

**Example**:
```
s = "catsand", start = 0

Found word = "cat" at [0:3]
Recursively solve s[3:] = "sand":
  → returns ["sand", "s and"]

Combine:
  "cat" + " " + "sand" = "cat sand"
  "cat" + " " + "s and" = "cat s and"

Found word = "cats" at [0:4]
Recursively solve s[4:] = "and":
  → returns ["and"]

Combine:
  "cats" + " " + "and" = "cats and"

memo[0] = ["cat sand", "cat s and", "cats and"]
```

---

### Base Cases

**Case 1: Reached end of string**
```python
if start == len(s):
    return [""]  # Empty sentence (valid segmentation complete)
```

Why return `[""]` not `[]`?
- `[""]` means "successfully segmented, no more words needed"
- `[]` means "no valid segmentation exists"
- When combining: `word + " " + ""` = `word` (correct)

**Case 2: No valid words from this position**
```python
results = []  # No words found
memo[start] = []
return []
```

---

### Optimization: Pre-check Feasibility

**Problem**: If no valid segmentation exists, backtracking wastes time.

**Solution**: Run Word Break I first
```python
def can_break(s, wordDict):
    dp = [False] * (len(s) + 1)
    dp[0] = True
    for i in range(1, len(s) + 1):
        for j in range(i):
            if dp[j] and s[j:i] in wordDict:
                dp[i] = True
                break
    return dp[len(s)]

if not can_break(s, wordDict):
    return []  # Early exit
```

**Time saved**: 
- Word Break I: O(n^2 × m) where m = avg word length
- If impossible, avoid exponential backtracking

---

### Example Walkthrough

**Input**: `s = "catsanddog"`, `wordDict = ["cat","cats","and","sand","dog"]`

```
dfs(0): "catsanddog"
  Try "cat" (match!) → dfs(3): "sanddog"
    Try "sand" (match!) → dfs(7): "dog"
      Try "dog" (match!) → dfs(10): ""
        Return [""]
      Return ["dog"]
    Return ["sand dog"]
  
  Try "cats" (match!) → dfs(4): "anddog"
    Try "and" (match!) → dfs(7): "dog"
      (memoized) Return ["dog"]
    Return ["and dog"]
  
  Return ["cat sand dog", "cats and dog"]
```

**Answer**: `["cat sand dog", "cats and dog"]`

---

### Time Complexity Analysis

**Worst case**: All possible segmentations

Example: `s = "aaaa"`, `wordDict = {"a", "aa", "aaa", "aaaa"}`

Number of segmentations:
- Length 1: 1 way
- Length 2: 2 ways ("a a", "aa")
- Length 3: 4 ways
- Length 4: 8 ways
- Length n: ~2^(n-1) ways (Fibonacci-like growth)

**Time**: O(2^n × n)
- 2^n possible segmentations
- n work to construct each sentence string

**With memoization**: Avoids redundant subtree exploration, but still must generate all results.

---

### Space Complexity

**Recursion depth**: O(n)
- At most n nested calls

**Memoization storage**: O(2^n × n)
- Store all possible segmentations
- Each segmentation is a string of length O(n)

**Total**: O(2^n × n)

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| Backtracking + memo | O(2^n × n) | O(2^n × n) | Optimal for generating all solutions |
| Pure backtracking | O(2^n × n) | O(n) | Recomputes subtrees |
| Trie + DFS | O(2^n × n) | O(total chars in dict) | Faster word lookup |
| BFS | O(2^n × n) | O(2^n × n) | More complex, no advantage |

In [ ]:
from typing import List

class Solution:
    def wordBreak(self, s: str, wordDict: List[str]) -> List[str]:
        """
        Backtracking with memoization to find all word segmentations.
        
        Time: O(2^n × n) in worst case
        Space: O(2^n × n) for memoization
        """
        word_set = set(wordDict)  # O(1) lookup
        memo = {}  # start_index -> list of segmentations
        
        def dfs(start):
            """
            Returns list of all valid segmentations for s[start:].
            """
            # Base case: reached end of string
            if start == len(s):
                return [""]  # Successfully segmented
            
            # Check memo
            if start in memo:
                return memo[start]
            
            results = []
            
            # Try all possible words starting from current position
            for end in range(start + 1, len(s) + 1):
                word = s[start:end]
                
                if word in word_set:
                    # Word found, recursively solve remaining
                    sub_sentences = dfs(end)
                    
                    # Combine current word with all sub-sentences
                    for sentence in sub_sentences:
                        if sentence:  # Non-empty sub-sentence
                            results.append(word + " " + sentence)
                        else:  # Empty sub-sentence (reached end)
                            results.append(word)
            
            memo[start] = results
            return results
        
        return dfs(0)

In [ ]:
# Test cases
tests = [
    ("catsanddog", ["cat","cats","and","sand","dog"], 
     ["cat sand dog", "cats and dog"]),
    ("pineapplepenapple", ["apple","pen","applepen","pine","pineapple"],
     ["pine apple pen apple", "pine applepen apple", "pineapple pen apple"]),
    ("catsandog", ["cats","dog","sand","and","cat"],
     []),  # No valid segmentation
    ("a", ["a"],
     ["a"]),
]

solver = Solution()
for s, wordDict, expected in tests:
    result = solver.wordBreak(s, wordDict)
    result_set = set(result)
    expected_set = set(expected)
    assert result_set == expected_set, f"Failed for s={s}: got {result}, expected {expected}"
print('All tests passed.')

## Complexity Analysis
- **Time**: O(2^n × n) worst case
  - Number of valid segmentations can be exponential (~2^n)
  - Each segmentation requires O(n) to construct string
  - Memoization avoids redundant subtree computation
- **Space**: O(2^n × n)
  - Memoization stores all possible segmentations
  - Each segmentation is O(n) characters
  - Recursion stack: O(n) depth

## Edge Cases & Pitfalls
- **No valid segmentation**: Return empty list
- **Single character**: Should work if in dictionary
- **Entire string is one word**: Return [s]
- **Repeated words**: "aaa" with dict ["a", "aa"] → multiple segmentations
- **Empty dictionary**: Return []
- **Common mistake**: Returning `[]` instead of `[""]` at base case (breaks combination)
- **Common mistake**: Not converting wordDict to set (O(n) lookup instead of O(1))
- **Common mistake**: String concatenation in loop (use join for efficiency)
- **Optimization**: Pre-check with Word Break I to avoid impossible cases

## Follow-up Variants
- **Limit word usage**: Each word can be used at most k times
- **Minimize spaces**: Return segmentation with fewest words
- **Weighted words**: Words have scores, maximize total score
- **Forbidden patterns**: Cannot use certain word combinations
- **Phonetic matching**: Words sound similar (fuzzy matching)
- **Multi-language**: Dictionary contains words from multiple languages
- **Stream processing**: String arrives character by character

## Takeaways
- **Memoization with backtracking** is essential for "find all solutions" problems.
- **Base case design matters**: `[""]` vs `[]` affects combination logic.
- Converting to set for **O(1) lookup** is crucial for performance.
- Understanding **exponential solution space** helps set expectations on complexity.
- **Pre-checking feasibility** (Word Break I) can save exponential work.
- This pattern extends to sentence parsing, code tokenization, and combinatorial generation.
- **String concatenation** in Python creates new objects; be mindful of overhead.
- Memoization doesn't reduce worst-case time (still exponential) but avoids redundant computation.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 139 | Word Break | DP, check if possible |
| LC 472 | Concatenated Words | Word Break variant |
| LC 425 | Word Squares | Backtracking with Trie |
| LC 131 | Palindrome Partitioning | Backtracking all partitions |
| LC 93 | Restore IP Addresses | Backtracking with constraints |
| LC 241 | Different Ways to Add Parentheses | Memoized recursion |